# ConvoBridge — FLAN-T5-small Meeting Summarization (Colab)

Lighter alternative to **Gemma** for meeting summarization.

| Item | Value |
|---|---|
| Base model | `google/flan-t5-small` |
| Method | LoRA (Seq2Seq) |
| Output | Google Drive `ConvoBridge/flan-t5-small-meeting/` |
| CPU RAM (infer) | ~1–1.5 GB |

**Runtime → Change runtime type → GPU (T4)**

This notebook is separate from Gemma training.


## Step 1 — Check GPU


In [ ]:
!nvidia-smi


## Step 2 — Install dependencies


In [ ]:
!pip install -q "transformers>=4.44.0" "peft>=0.13.0" "datasets>=2.20.0" \
  "accelerate>=0.33.0" huggingface_hub pyyaml evaluate


## Step 3 — Mount Google Drive


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/ConvoBridge")
OUTPUT_DIR = DRIVE_ROOT / "flan-t5-small-meeting"
CHECKPOINT_DIR = DRIVE_ROOT / "flan_t5_checkpoints"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive ready:", DRIVE_ROOT)


## Step 4 — Config


In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> GPU"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

BASE_MODEL = "google/flan-t5-small"

PROMPT_TEMPLATE = """You are ConvoBridge Meeting Assistant. Summarize the meeting transcript below.

Return ONLY the following sections (no extra text):

SUMMARY:
<2-4 sentence overview>

KEY_POINTS:
- <bullet 1>
- <bullet 2>
- <bullet 3>

ACTION_ITEMS:
- <owner or team>: <task> (due: <if mentioned, else TBD>)

Rules:
- Use only information from the transcript.
- If no action items exist, write: ACTION_ITEMS:\n- None

TRANSCRIPT:
{transcript}
"""

TRAIN_CFG = {
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 3e-4,
    "num_epochs": 3,
    "per_device_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "max_source_length": 512,
    "max_target_length": 256,
    "save_steps": 200,
    "max_train_rows": 8000,
}

print("BASE_MODEL:", BASE_MODEL)
print("OUTPUT_DIR:", OUTPUT_DIR)


## Step 5 — Build training data (SamSum online)

Creates the same structured SUMMARY / KEY_POINTS / ACTION_ITEMS format used by ConvoBridge.


In [ ]:
import json
import re
from datasets import load_dataset

def wrap_summary(summary: str) -> str:
    summary = summary.strip()
    sentences = re.split(r"(?<=[.!?])\s+", summary)
    key_points = [s.strip() for s in sentences if len(s.strip()) > 10][:5]
    if not key_points:
        key_points = [summary]
    kp = "\n".join(f"- {k}" for k in key_points)
    return (
        f"SUMMARY:\n{summary}\n\n"
        f"KEY_POINTS:\n{kp}\n\n"
        f"ACTION_ITEMS:\n- None"
    )

print("Loading knkarthick/samsum ...")
raw = load_dataset("knkarthick/samsum", split="train")
rows = []
for ex in raw:
    dialogue = (ex.get("dialogue") or "").strip()
    summary = (ex.get("summary") or "").strip()
    if len(dialogue) < 40 or len(summary) < 10:
        continue
    rows.append({
        "instruction": "Summarize the meeting transcript. Return SUMMARY, KEY_POINTS, and ACTION_ITEMS.",
        "input": dialogue,
        "output": wrap_summary(summary),
        "source": "samsum",
    })
    if len(rows) >= TRAIN_CFG["max_train_rows"]:
        break

DATA_PATH = Path("/content/train_meeting_summary.jsonl")
with DATA_PATH.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(rows)} rows -> {DATA_PATH}")
print("Example output preview:\n", rows[0]["output"][:300])


## Step 6 — Train FLAN-T5-small + LoRA


In [ ]:
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

lora = LoraConfig(
    r=TRAIN_CFG["lora_r"],
    lora_alpha=TRAIN_CFG["lora_alpha"],
    lora_dropout=TRAIN_CFG["lora_dropout"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q", "v"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

ds = load_dataset("json", data_files={"train": str(DATA_PATH)})["train"]
print("Train size:", len(ds))

def tokenize(batch):
    sources = [PROMPT_TEMPLATE.format(transcript=t.strip()) for t in batch["input"]]
    model_inputs = tokenizer(
        sources,
        max_length=TRAIN_CFG["max_source_length"],
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch["output"],
        max_length=TRAIN_CFG["max_target_length"],
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=TRAIN_CFG["num_epochs"],
    per_device_train_batch_size=TRAIN_CFG["per_device_batch_size"],
    gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
    learning_rate=TRAIN_CFG["learning_rate"],
    logging_steps=20,
    save_steps=TRAIN_CFG["save_steps"],
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=TRAIN_CFG["max_target_length"],
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    remove_unused_columns=False,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
    processing_class=tokenizer,
)

trainer.train()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved ->", OUTPUT_DIR)
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))


## Step 7 — Quick inference test


In [ ]:
from peft import PeftModel

sample = """Alice: We need to finish the backend by Friday.
Bob: I will handle the deployment.
Alice: Great, Sara will own the UI. Let's sync Monday at 10 AM."""

tok = AutoTokenizer.from_pretrained(str(OUTPUT_DIR))
base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
mdl = PeftModel.from_pretrained(base, str(OUTPUT_DIR))
mdl.cuda().eval()

src = PROMPT_TEMPLATE.format(transcript=sample)
inputs = tok(src, return_tensors="pt", truncation=True, max_length=512).to("cuda")
with torch.no_grad():
    out = mdl.generate(**inputs, max_new_tokens=256, num_beams=4)
print(tok.decode(out[0], skip_special_tokens=True))


## Step 8 — Download to your laptop

1. In Google Drive open `MyDrive/ConvoBridge/flan-t5-small-meeting/`
2. Download the folder
3. Copy into your project:

```text
summarization/models/flan-t5-small-meeting/
```

Backend wiring to this model is a separate step (ask when ready).
